# XXXXXXXXXXXX
# By Mohamed Eltayeb

# Import libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
import optuna
tqdm.pandas()

from pandas.plotting import scatter_matrix
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, KFold, StratifiedGroupKFold, GroupKFold
from sklearn.model_selection import TimeSeriesSplit

from lightgbm import LGBMRegressor, LGBMClassifier
from xgboost import XGBRegressor, XGBClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import VotingRegressor, VotingClassifier, BaggingRegressor, BaggingClassifier 
from sklearn.ensemble import StackingRegressor, StackingClassifier
from sklearn.compose import TransformedTargetRegressor

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from openTSNE import TSNE
import category_encoders as ce
from sklearn.metrics import mean_absolute_error
!pip install textstat
import textstat
# !pip install -qq reverse_geocoder
import reverse_geocoder as rg
# !pip install lofo-importance
from lofo import LOFOImportance, Dataset, plot_importance


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
#pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.rcParams["figure.figsize"] = (12, 8)
pd.set_option('display.max_columns', None)

In [ ]:
#Group Time Series Split
from sklearn.model_selection._split import _BaseKFold, indexable, _num_samples
from sklearn.utils.validation import _deprecate_positional_args

class GroupTimeSeriesSplit(_BaseKFold):
    @_deprecate_positional_args
    def __init__(self,n_splits=5,*,max_train_size=None):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.max_train_size = max_train_size

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set.
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training data, where n_samples is the number of samples
            and n_features is the number of features.
        y : array-like of shape (n_samples,)
            Always ignored, exists for compatibility.
        groups : array-like of shape (n_samples,)
            Group labels for the samples used while splitting the dataset into
            train/test set.
        Yields
        ------
        train : ndarray
            The training set indices for that split.
        test : ndarray
            The testing set indices for that split.
        """
        if groups is None:
            raise ValueError(
                "The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)
        n_splits = self.n_splits
        n_folds = n_splits + 1
        group_dict = {}
        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_samples = _num_samples(X)
        n_groups = _num_samples(unique_groups)
        for idx in np.arange(n_samples):
            if (groups[idx] in group_dict):
                group_dict[groups[idx]].append(idx)
            else:
                group_dict[groups[idx]] = [idx]
        if n_folds > n_groups:
            raise ValueError(
                ("Cannot have number of folds={0} greater than"
                 " the number of groups={1}").format(n_folds,
                                                     n_groups))
        group_test_size = n_groups // n_folds
        group_test_starts = range(n_groups - n_splits * group_test_size,
                                  n_groups, group_test_size)
        for group_test_start in group_test_starts:
            train_array = []
            test_array = []
            for train_group_idx in unique_groups[:group_test_start]:
                train_array_tmp = group_dict[train_group_idx]
                train_array = np.sort(np.unique(
                                      np.concatenate((train_array,
                                                      train_array_tmp)),
                                      axis=None), axis=None)
            train_end = train_array.size
            if self.max_train_size and self.max_train_size < train_end:
                train_array = train_array[train_end -
                                          self.max_train_size:train_end]
            for test_group_idx in unique_groups[group_test_start:
                                                group_test_start +
                                                group_test_size]:
                test_array_tmp = group_dict[test_group_idx]
                test_array = np.sort(np.unique(
                                              np.concatenate((test_array,
                                                              test_array_tmp)),
                                     axis=None), axis=None)
            yield [int(i) for i in train_array], [int(i) for i in test_array]

In [ ]:
#Plot the Features Importances
def plotImp(model, X , num = 20, fig_size = (40, 20)):
    feature_imp = pd.DataFrame({'Value':model.feature_importances_,'Feature':X.columns})
    plt.figure(figsize=fig_size)
    sns.set(font_scale = 5)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", 
                                                        ascending=False)[0:num])
    plt.title('Features (avg over folds)')
    plt.tight_layout()
    plt.savefig('importances-01.png')
    plt.show()
    sns.set()

In [ ]:
#Reduce Memory Usage
def reduce_memory_usage(df):
    
    for col in df.columns:
        col_type = df[col].dtype.name
        if ((col_type != 'datetime64[ns]') & (col_type != 'category')):
            if (col_type != 'object'):
                c_min = df[col].min()
                c_max = df[col].max()

                if str(col_type)[:3] == 'int':
                    if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df[col] = df[col].astype(np.int8)
                    elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df[col] = df[col].astype(np.int16)
                    elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df[col] = df[col].astype(np.int32)
                    elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        df[col] = df[col].astype(np.int64)

                else:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
                    else:
                        pass
            else:
                df[col] = df[col].astype('category')
    
    return df

# Read the training and testing data


In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
store_df = pd.read_csv("store.csv")

In [ ]:
train_df = reduce_memory_usage(train_df)
test_df = reduce_memory_usage(test_df)

## Reduce Memory Usage for The Training IDs

In [ ]:
#Use if The Data is Very Big
# train_df['ID'] = train_df['ID'].str[-16:].str.hex_to_int().astype('int64')  #For cudf
train_df['ID'] = train_df['ID'].apply(lambda x: int(x[-16:],16) ).astype('int64')  #For pandas

## Store Target Labels For Later Usage (For Proabilities Predictions Only)

In [ ]:
Target_Values = train_df['Target'].value_counts().keys().sort_values()

# Add Temporal Features

In [ ]:
train_df = train_df.sort_values('Date').reset_index(drop=True) 
test_df = test_df.sort_values('Date').reset_index(drop=True)

#Check the original order and remember to get it back by the reversed command
#If you have Lags, Rollings or Fourier Frequencies you should order then get back to the original order
#Copy this code after the Lags, Rollings and Fourier definitions and modify it accordingly
# train_df.sort_values(["Store"], ignore_index=True, inplace=True)
# test_df.sort_values(["Store"], ignore_index=True, inplace=True)
# train_df.sort_values(["Year","Month","Day"], ascending=False ,ignore_index=True, inplace=True)
# test_df.sort_values(["Year","Month","Day"], ascending=False ,ignore_index=True, inplace=True)

train_df['Date'] = train_df['Date'].astype('datetime64[ns]')
test_df['Date'] = test_df['Date'].astype('datetime64[ns]')

for dataset in (train_df,test_df):
    dataset['Date_Int'] = dataset['Date'].astype(np.int64) * 1e-9
    dataset['Day'] = dataset.Date.dt.day
    dataset['Month'] = dataset.Date.dt.month
    dataset['Year'] = dataset.Date.dt.year         
    dataset['Year'] = (dataset.Date.dt.year - 2000).astype('int8')     #Memory Efficient
    dataset['Hour'] = dataset.Date.dt.hour
    dataset['Minute'] = dataset.Date.dt.minute
    dataset['Second'] = dataset.Date.dt.second
    dataset['DayOfWeek'] = dataset.Date.dt.dayofweek
    dataset['DayOfYear'] = dataset.Date.dt.dayofyear
    dataset['WeekOfYear'] = dataset.Date.dt.weekofyear
    dataset['Quarter'] = dataset.Date.dt.quarter
    dataset.set_index('Date', inplace=True)

In [ ]:
ID = test_df['ID']
train_df.drop('ID',inplace=True,axis=1)
test_df.drop('ID',inplace=True,axis=1)

# Build a Quick Baseline

In [ ]:
feats = list(train_df.select_dtypes(include=['object','category']).columns)
# feats.remove('Target')
le = LabelEncoder()
df = pd.concat([train_df, test_df])
for f in feats:
    le.fit(df[f])
    train_df[f] = le.transform(train_df[f])
    test_df[f] = le.transform(test_df[f])

In [ ]:
from lightgbm import LGBMRegressor
from lightgbm import LGBMClassifier

X = train_df.copy()
y = X['Target']
X.drop('Target',inplace=True,axis=1)

lg = LGBMRegressor(max_depth=6)
# lg = LGBMClassifier(max_depth=6)
lg.fit(X,y)

plotImp(lg,X)

test_df['Target'] = lg.predict(test_df)
submission = pd.DataFrame({"Id": ID ,"Target": test_df.Target})
submission.to_csv('Baseline.csv',index=False)

# Exploratory data analysis

#### EDA:
* Seasonality (monthly, daily, etc.)
* Trends (big for proper evaluation)
* Autocorrelation
* Diff. between older and newer series
* Tons of raw samples 
* Difference between training and testing histograms

# Features:

* Id - an Id that represents a (Store, Date) duple within the test set
* Store - a unique Id for each store
* ...

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

In [ ]:
#The cardinality of each catgorical feature (Training)
cat_cols = train_df.columns
for col in cat_cols:
    print(col, train_df[col].nunique())

In [ ]:
#The cardinality of each catgorical feature (Testing)
cat_cols = test_df.columns
for col in cat_cols:
    print(col, test_df[col].nunique())

# Data Preprocessing

In [ ]:
# Drop Duplicates
print('Samples Before Dropping: ', train_df.shape[0])
train_df = train_df.drop_duplicates()
print('Samples After Dropping: ', train_df.shape[0])

# Feature Engineering

#### Features:
* Difference Between dates
* Features clusters then statistics for each cluster (Using correlation heatmap or plot statistics against index)
* If there is a relation between two features, add difference or ratio
* Interaction Features
* clustering these time-series based on the performance of the best model. Then training different models for each cluster.

## Lags and Rolling For Features With Respect To Another Feature (Store,Clients...etc)

In [ ]:
feats = ['Feature_1','Feature_2']
for dataset in (train_df,test_df):
    for feat in feats:
        for window in [1,5,10,20,40,60,100]:
            dataset[f'{feat}_Lag_{window}'] = dataset['feature'].map(dict(dataset.groupby('feature')[feat].shift(window)))
            dataset[f'{feat}_Dif_{window}'] = dataset['feature'].map(dict(dataset.groupby('feature')[feat].dif(window)))
            dataset[f'{feat}_Rol_{window}'] = dataset['feature'].map(dict(dataset.groupby('feature')[feat].rolling(window)))

## Lags and Rolling for Target with Respect to Another Feature

In [ ]:
df = pd.concat([train_df,test_df])
df['date'] = df.index
feat = 'Target'
Pivots_df = {} 
Pivots_df[feat] = reduce_memory_usage(pd.pivot_table(df, values = feat, index = "date", columns = "Store/Client/Secrities").ffill())
print(f'{feat} Done.')

In [ ]:
for func in [lag, dif, rol]:
    for window in [100,200]: #Should be > test_df days
        df = reduce_memory_usage(pd.merge(df, func(window,Pivots_df[feat],feat), on = ["date","Store/Client/Secrities"], how = "left"))
        print(f'Target {window} Done')

df.set_index("date", inplace=True)
train_df = df[:train_df.shape[0]]
test_df = df[train_df.shape[0]:]
test_df.drop(feat,inplace=True,axis=1)  #Drop any other feature that doesn't exist in test_df

## Lags Features For Features Other Than The Target

In [ ]:
# ACF between s_0 and s_3 = relation between s_0, s_1, s_2, s_3
# PACF between s_0 and s_3 = relation between s_0, s_3
import statsmodels.tsa.api as smt
import statsmodels.api as sm

def tsplot(y, lags=60, figsize=(12, 7), style='bmh'):
     if not isinstance(y, pd.Series):
        y = pd.Series(y)
        
     with plt.style.context(style):    
        fig = plt.figure(figsize=figsize)
        layout = (2, 2)
        ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
        acf_ax = plt.subplot2grid(layout, (1, 0))
        pacf_ax = plt.subplot2grid(layout, (1, 1))
        
        y.plot(ax=ts_ax)
        p_value = sm.tsa.stattools.adfuller(y)[1]
        ts_ax.set_title('Time Series Analysis Plots\n Dickey-Fuller: p={0:.5f}'.format(p_value))
        smt.graphics.plot_acf(y, lags=lags, ax=acf_ax)
        smt.graphics.plot_pacf(y, lags=lags, ax=pacf_ax)
        plt.tight_layout()
tsplot(train_df['Feature'].fillna(0))

In [ ]:
feats = ['Feature_1','Feature_2']
for dataset in (train_df,test_df):
    for feat in feats:
        for window in [1,5,10,20,40,60,100]:
            dataset[f'{feat}_Lag_{window}'] = dataset[feat].shift(window)
            dataset[f'{feat}_Dif_{window}'] = dataset[feat].diff(window)

## Lags For Target

In [ ]:
df = pd.concat([train_df,test_df])
feat = 'Target'
for window in [1,5,10,20,40,60,100]:
    df[f'{feat}_Lag_{window}'] = df[feat].shift(window)
    df[f'{feat}_Dif_{window}'] = df[feat].diff(window)

train_df = df[:train_df.shape[0]]
test_df = df[train_df.shape[0]:]
test_df.drop(feat,inplace=True,axis=1)  #Drop any other feature that doesn't exist in test_df

## Rolling Features For Features Other Than The Target (Numerical)

In [ ]:
def rolling(feature,window):
    for dataset in (train_df,test_df):
        rol_df = dataset.rolling(window)[feature]
        dataset[f"{feature}_rolling_mean_{window}"] = rol_df.mean()
        dataset[f"{feature}_rolling_max_{window}"] = rol_df.max()
        dataset[f"{feature}_rolling_min_{window}"] = rol_df.min()
        dataset[f"{feature}_rolling_std_{window}"] = rol_df.std()
        dataset[f"{feature}_rolling_sum_{window}"] = rol_df.sum()
        dataset[f"{feature}_rolling_range_{window}"] = dataset[f"{feature}_rolling_max_{window}"] - dataset[f"{feature}_rolling_min_{window}"]
        dataset[f"{feature}_rolling_kurt_{window}"] = rol_df.kurt()
        dataset[f"{feature}_rolling_skew_{window}"] = rol_df.skew()
        dataset[f"{feature}_rolling_qua10_{window}"] = rol_df.quantile(0.10)
        dataset[f"{feature}_rolling_qua25_{window}"] = rol_df.quantile(0.25)
        dataset[f"{feature}_rolling_qua75_{window}"] = rol_df.quantile(0.75)
        dataset[f"{feature}_rolling_qua90_{window}"] = rol_df.quantile(0.90)
        print(f'{feature} {window} Done!')

feats = ['Feature_1','Feature_2']
for feat in feats:
    for window in [1,5,10,20,40,60,100]:
        rolling(feat,window)

## Rolling Features For Target (Numerical)

In [ ]:
def rolling(feature,window):
    for dataset in (train_df,test_df):
        rol_df = train_df.rolling(window)[feature]
        dataset[f"{feature}_rolling_mean_{window}"] = rol_df.mean()
        dataset[f"{feature}_rolling_max_{window}"] = rol_df.max()
        dataset[f"{feature}_rolling_min_{window}"] = rol_df.min()
        dataset[f"{feature}_rolling_std_{window}"] = rol_df.std()
        dataset[f"{feature}_rolling_sum_{window}"] = rol_df.sum()
        dataset[f"{feature}_rolling_range_{window}"] = dataset[f"{feature}_rolling_min_{window}"] - dataset[f"{feature}_rolling_min_{window}"]
        dataset[f"{feature}_rolling_kurt_{window}"] = rol_df.kurt()
        dataset[f"{feature}_rolling_skew_{window}"] = rol_df.skew()
        dataset[f"{feature}_rolling_qua10_{window}"] = rol_df.quantile(0.10)
        dataset[f"{feature}_rolling_qua25_{window}"] = rol_df.quantile(0.25)
        dataset[f"{feature}_rolling_qua75_{window}"] = rol_df.quantile(0.75)
        dataset[f"{feature}_rolling_qua90_{window}"] = rol_df.quantile(0.90)
        print(f'{feature} {window} Done!')

for window in [1,5,10,20,40,60,100]:
    rolling('Target',window)

## Rolling Features For Features Other Than The Target (Categorical) (Slow for Big Dataset)

In [ ]:
feats = ['Feature_1','Feature_2']
le = LabelEncoder()
df = pd.concat([train_df,test_df])
for feat in feats:
    le.fit(df[feat])
    train_df[feat] = le.transform(train_df[feat])
    test_df[feat] = le.transform(test_df[feat])

def rolling(feature,window):
    for dataset in (train_df,test_df): 
        rol_df = dataset.rolling(window)[feature]
        dataset[f"{feature}_rolling_mode_{window}"] = rol_df.agg(lambda x: pd.Series.mode(x)[0])
        dataset[f"{feature}_rolling_nunique_{window}"] = rol_df.agg(pd.Series.nunique)
        print(f'{feature} {window} Done!')

for feat in feats:
    for window in [1,5,10,20,40,60,100]:
        rolling(feat,window)

## Rolling Features For Target (Categorical) (Slow for Big Dataset)

In [ ]:
feat = 'Target'

le = LabelEncoder()
df = pd.concat([train_df,test_df])
le.fit(df[feat])
df[feat] = le.transform(df[feat])

def rolling(feature,window):
        rol_df = df.rolling(window)[feature]
        dataset[f"{feature}_rolling_mode_{window}"] = rol_df.agg(lambda x: pd.Series.mode(x)[0])
        dataset[f"{feature}_rolling_nunique_{window}"] = rol_df.agg(pd.Series.nunique)
        print(f'{feature} {window} Done!')

for window in [1,5,10,20,40,60,100]:
       rolling(feat,window)

train_df = df[:train_df.shape[0]]
test_df = df[train_df.shape[0]:]
test_df.drop(feat,inplace=True,axis=1)  #Drop any other feature that doesn't exist in test_df

## Statistics of Two Features 

In [ ]:
def Statistics(features):
    for dataset in (train_df,test_df):
        dataset[f"SD_features"] = dataset[features].std(axis=1)
        dataset[f"MEAN_features"] = dataset[features].mean(axis=1)
        dataset[f"MIN_features"] = dataset[features].min(axis=1)
        dataset[f"MAX_features"] = dataset[features].max(axis=1)
        dataset[f"MEAN_ABS_CHANGE_features"] = np.abs(dataset[features].diff().mean(axis=1))
        dataset[f"RANGE_features"] = dataset[f"MAX_features"] - dataset[f"MIN_features"]
        dataset[f"RATIO_features"] = dataset[f"MIN_features"] / dataset[f"MAX_features"]
        dataset[f"SUM_features"] = dataset[features].sum(axis=1)
        dataset[f"EQUAL_features"] = dataset[f"MAX_features"] == dataset[f"MIN_features"]
        
feats = ['Feature_1','Feature_2']        
Statistics(feats)

## Project cyclic features into unit circle 

In [ ]:
for dataset in (train_df,test_df):
    dataset['Month_Sin'] = np.sin(2*np.pi*dataset['Month']/12)
    dataset['Month_Cos'] = np.cos(2*np.pi*dataset['Month']/12)
    dataset['Day_Sin'] = np.sin(2*np.pi*dataset['Day']/30)
    dataset['Day_Cos'] = np.cos(2*np.pi*dataset['Day']/30)
    dataset['Hour_Sin'] = np.sin(2*np.pi*dataset['Hour']/24)
    dataset['Hour_Cos'] = np.cos(2*np.pi*dataset['Hour']/24)
    dataset['Minute_Sin'] = np.sin(2*np.pi*dataset['Minute']/60)
    dataset['Minute_Cos'] = np.cos(2*np.pi*dataset['Minute']/60)
    dataset['Second_Sin'] = np.sin(2*np.pi*dataset['Second']/60)
    dataset['Second_Cos'] = np.cos(2*np.pi*dataset['Second']/60)

## Combination Between Time Features

In [ ]:
for dataset in (train_df,test_df):
    dataset['Year_Month'] = dataset['Year'].astype(str) + '-' + dataset['Month'].astype(str)
    dataset['Year_Week'] = dataset['Year'].astype(str) + '-' + dataset['WeekOfYear'].astype(str)
    dataset['Year_Month_Day'] = dataset['Year'].astype(str) + '-' + dataset['Month'].astype(str) + '-' + dataset['Day'].astype(str)
    dataset['Year_Month_Day_Hour']  = dataset['Year'].astype(str) + '-' + dataset['Month'].astype(str) + '-' + dataset['Day'].astype(str) + '-' + dataset['Hour'].astype(str)
    dataset['Year_Month_Day_Hour_Minute']  = dataset['Year'].astype(str) + '-' + dataset['Month'].astype(str) + '-' + dataset['Day'].astype(str) + '-' + dataset['Hour'].astype(str) + '-' + dataset['Minute'].astype(str)
    dataset['Year_Month_Day_Hour_Minute_Seconds']  = dataset['Year'].astype(str) + '-' + dataset['Month'].astype(str) + '-' + dataset['Day'].astype(str) + '-' + dataset['Hour'].astype(str) + '-' + dataset['Minute'].astype(str) + '-' + dataset['Second'].astype(str)
    dataset['Month_Day'] = dataset['Month'].astype(str) + '-' + dataset['Day'].astype(str)
    dataset['Day_Hour'] = dataset['Day'].astype(str) + '-' + dataset['Hour'].astype(str)
    dataset['Hour_Minute'] = dataset['Hour'].astype(str) + '-' + dataset['Minute'].astype(str)
    dataset['Hour_Minute_Seconds'] = dataset['Hour'].astype(str) + '-' + dataset['Minute'].astype(str) + '-' + dataset['Second'].astype(str)
#Label Encode Them
feats = ['Year_Month','Year_Week','Year_Month_Day',
         'Year_Month_Day_Hour','Year_Month_Day_Hour_Minute',
         'Year_Month_Day_Hour_Minute_Seconds', 'Month_Day',
         'Day_Hour','Hour_Minute','Hour_Minute_Seconds']

le = LabelEncoder()
df = pd.concat([train_df,test_df])

for feat in feats:
    le.fit(df[feat])
    train_df[feat] = le.transform(train_df[feat])
    test_df[feat] = le.transform(test_df[feat])

## Add Aggregations Based on Time With NO Target

In [ ]:
def nth_largest(n, s):
    return s.nlargest(n).iloc[-1] if len(s) >= n else np.nan

In [ ]:
def Agg(Feature):
    for dataset in (train_df,test_df):
        for feat_1 in ['Year','Month','WeekOfYear','Month_Day','Day','Day_Hour','Hour','Hour_Minute','Hour_Minute_Seconds']:
            dataset[f'{Feature}_Agg_{feat_1}_sem'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].sem()))
            dataset[f'{Feature}_Agg_{feat_1}_cumsum'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].cumsum()))
            dataset[f'{Feature}_Agg_{feat_1}_mean'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].mean()))
            dataset[f'{Feature}_Agg_{feat_1}_median'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].median()))
            dataset[f'{Feature}_Agg_{feat_1}_std'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].std())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_min'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].min()))
            dataset[f'{Feature}_Agg_{feat_1}_max'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].max()))
            dataset[f'{Feature}_Agg_{feat_1}_2nd_max'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].apply(lambda s: nth_largest(2, s))))
            dataset[f'{Feature}_Agg_{feat_1}_3rd_max'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].apply(lambda s: nth_largest(3, s))))
            dataset[f'{Feature}_Agg_{feat_1}_sum'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].sum()))
            dataset[f'{Feature}_Agg_{feat_1}_range'] = dataset[f'{Feature}_Agg_{feat_1}_max'] - dataset[f'{Feature}_Agg_{feat_1}_min']
            dataset[f'{Feature}_Agg_{feat_1}_var'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].var())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_last'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].last()))
            dataset[f'{Feature}_Agg_{feat_1}_first'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].first()))
            dataset[f'{Feature}_Agg_{feat_1}_skew'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].skew()))
            dataset[f'{Feature}_Agg_{feat_1}_kurt'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].kurt()))
            for n in [0.10,0.25,0.75,0.90]:
                dataset[f'{Feature}_Agg_{feat_1}_quantile_{n}'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].quantile(n)))        
feats = ['Feat_1', 'Feat_2']
for feat in feats:         
    Agg(feat)

## Add Aggregations Based on Time WITH Target

In [ ]:
def Agg(Feature):
    for dataset in (train_df,test_df):
        for feat_1 in ['Year','Month','WeekOfYear','Month_Day','Day','Day_Hour','Hour','Hour_Minute','Hour_Minute_Seconds']:
            dataset[f'{Feature}_Agg_{feat_1}_sem'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].sem()))
            dataset[f'{Feature}_Agg_{feat_1}_cumsum'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].cumsum()))
            dataset[f'{Feature}_Agg_{feat_1}_mean'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].mean()))
            dataset[f'{Feature}_Agg_{feat_1}_median'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].median()))
            dataset[f'{Feature}_Agg_{feat_1}_std'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].std())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_min'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].min()))
            dataset[f'{Feature}_Agg_{feat_1}_max'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].max()))
            dataset[f'{Feature}_Agg_{feat_1}_2nd_max'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].apply(lambda s: nth_largest(2, s))))
            dataset[f'{Feature}_Agg_{feat_1}_3rd_max'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].apply(lambda s: nth_largest(3, s))))
            dataset[f'{Feature}_Agg_{feat_1}_sum'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].sum()))
            dataset[f'{Feature}_Agg_{feat_1}_range'] = dataset[f'{Feature}_Agg_{feat_1}_max'] - dataset[f'{Feature}_Agg_{feat_1}_min']
            dataset[f'{Feature}_Agg_{feat_1}_var'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].var())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_last'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].last()))
            dataset[f'{Feature}_Agg_{feat_1}_first'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].first()))
            dataset[f'{Feature}_Agg_{feat_1}_skew'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].skew()))
            dataset[f'{Feature}_Agg_{feat_1}_kurt'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].kurt()))
            for n in [0.10,0.25,0.75,0.90]:
                dataset[f'{Feature}_Agg_{feat_1}_quantile_{n}'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].quantile(n)))        
feats = ['Feat_1', 'Feat_2']
for feat in feats:         
    Agg(feat)

## Add Aggregations NOT Based on Time WITH NO Target (Numerical)

In [ ]:
def Agg(Feature):
    for dataset in (train_df,test_df):
        for feat_1 in ['Feat_1','Feat_2']:
            dataset[f'{Feature}_Agg_{feat_1}_sem'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].sem()))
            dataset[f'{Feature}_Agg_{feat_1}_cumsum'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].cumsum()))
            dataset[f'{Feature}_Agg_{feat_1}_mean'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].mean()))
            dataset[f'{Feature}_Agg_{feat_1}_median'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].median()))
            dataset[f'{Feature}_Agg_{feat_1}_std'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].std())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_min'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].min()))
            dataset[f'{Feature}_Agg_{feat_1}_max'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].max()))
            dataset[f'{Feature}_Agg_{feat_1}_2nd_max'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].apply(lambda s: nth_largest(2, s))))
            dataset[f'{Feature}_Agg_{feat_1}_3rd_max'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].apply(lambda s: nth_largest(3, s))))
            dataset[f'{Feature}_Agg_{feat_1}_sum'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].sum()))
            dataset[f'{Feature}_Agg_{feat_1}_range'] = dataset[f'{Feature}_Agg_{feat_1}_max'] - dataset[f'{Feature}_Agg_{feat_1}_min']
            dataset[f'{Feature}_Agg_{feat_1}_var'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].var())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_last'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].last()))
            dataset[f'{Feature}_Agg_{feat_1}_first'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].first()))
            dataset[f'{Feature}_Agg_{feat_1}_skew'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].skew()))
            dataset[f'{Feature}_Agg_{feat_1}_kurt'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].kurt()))
            for n in [0.10,0.25,0.75,0.90]:
                dataset[f'{Feature}_Agg_{feat_1}_quantile_{n}'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].quantile(n)))        
feats = ['Feat_1', 'Feat_2']
for feat in feats:         
    Agg(feat)       #If applicable, try to take ratios/product between the aggs of the two feats

## Add Aggregations NOT Based on Time WITH Target (Numerical)

In [ ]:
def Agg(Feature):
    for dataset in (train_df,test_df):
        for feat_1 in ['Feat_1','Feat_2']:
            dataset[f'{Feature}_Agg_{feat_1}_sem'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].sem()))
            dataset[f'{Feature}_Agg_{feat_1}_cumsum'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].cumsum()))
            dataset[f'{Feature}_Agg_{feat_1}_mean'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].mean()))
            dataset[f'{Feature}_Agg_{feat_1}_median'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].median()))
            dataset[f'{Feature}_Agg_{feat_1}_std'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].std())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_min'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].min()))
            dataset[f'{Feature}_Agg_{feat_1}_max'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].max()))
            dataset[f'{Feature}_Agg_{feat_1}_2nd_max'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].apply(lambda s: nth_largest(2, s))))
            dataset[f'{Feature}_Agg_{feat_1}_3rd_max'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].apply(lambda s: nth_largest(3, s))))
            dataset[f'{Feature}_Agg_{feat_1}_sum'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].sum()))
            dataset[f'{Feature}_Agg_{feat_1}_range'] = dataset[f'{Feature}_Agg_{feat_1}_max'] - dataset[f'{Feature}_Agg_{feat_1}_min']
            dataset[f'{Feature}_Agg_{feat_1}_var'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].var())).fillna(0)
            dataset[f'{Feature}_Agg_{feat_1}_last'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].last()))
            dataset[f'{Feature}_Agg_{feat_1}_first'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].first()))
            dataset[f'{Feature}_Agg_{feat_1}_skew'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].skew()))
            dataset[f'{Feature}_Agg_{feat_1}_kurt'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].kurt()))
            for n in [0.10,0.25,0.75,0.90]:
                dataset[f'{Feature}_Agg_{feat_1}_quantile_{n}'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].quantile(n)))        
Agg('Target')

## Add Aggregations NOT Based on Time WITH NO Target (Categorical)

In [ ]:
def Agg(Feature):
    for dataset in (train_df,test_df):
        for feat_1 in ['Feat_1','Feat_2']:
            dataset[f'{Feature}_Agg_{feat_1}_mode'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].agg(lambda x: pd.Series.mode(x)[0])))
            dataset[f'{Feature}_Agg_{feat_1}_nunique'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].nunique()))
            dataset[f'{Feature}_Agg_{feat_1}_count'] = dataset[feat_1].map(dict(dataset.groupby(feat_1)[Feature].count()))
                
feats = ['Feat_1', 'Feat_2']
for feat in feats:         
    Agg(feat)

## Add Aggregations NOT Based on Time WITH Target (Categorical)

In [ ]:
def Agg(Feature):
    for dataset in (train_df,test_df):
        for feat_1 in ['Feat_1','Feat_2']:
            dataset[f'{Feature}_Agg_{feat_1}_mode'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].agg(lambda x: pd.Series.mode(x)[0])))
            dataset[f'{Feature}_Agg_{feat_1}_nunique'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].nunique()))
            dataset[f'{Feature}_Agg_{feat_1}_count'] = dataset[feat_1].map(dict(train_df.groupby(feat_1)[Feature].count()))

                
Agg('Target')

## Foureier Frequnecies and Amplitudes (Only for Numeric Features)

In [ ]:
df_fourier = train_df.loc[train_df['feat']==2015]['Sales']  #feat is a feature that may have seasonality e.g. store, year, month...etc

Y = np.fft.fft(df_fourier.values)
freq = np.fft.fftfreq(len(Y), 1)
n = len(freq)
plt.figure()
plt.plot( freq[:int(n/2)], np.abs(Y)[:int(n/2)] )
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.show()

#### For Target Features (Mapping into Test) 

In [ ]:
import pandas as pd
import numpy as np

# Specify the number of features you want to create
NUM_FEATURES = 3

def calculate_features(group, feat_1, feat_2):
    result = {}
    Y = np.fft.fft(group[feat_2].values)
    Y = np.abs(Y)
    freq = np.fft.fftfreq(len(Y), 1)

    Y = np.delete(Y, np.argmax(Y))  # delete the intercept frequency
    freq = np.delete(freq, np.argmax(Y))  # delete the intercept frequency
    
    for i in range(NUM_FEATURES):
        amp_index = np.argmax(Y)
        amplitude = Y[amp_index]
        frequency = freq[amp_index]
        
        result[f'Amplitude_{i+2}_{feat_1}_{feat_2}'] = amplitude
        result[f'Frequency_{i+2}_{feat_1}_{feat_2}'] = frequency
        
        Y = np.delete(Y, amp_index)
        freq = np.delete(freq, amp_index)
        
    return pd.Series(result)

for feat_1 in ['Month']:
    for feat_2 in ['H2O Pressure (cm)']:
        features_df = train_df.groupby(feat_1).apply(lambda group: calculate_features(group, feat_1, feat_2)).reset_index()
        train_df = pd.merge(train_df, features_df, on=feat_1)

        # Mapping the same features to the test set
        test_df = pd.merge(test_df, features_df, on=feat_1, how='left')

train_df.sort_values('Date',inplace=True)
test_df.sort_values('Date',inplace=True)

### For Other Features

In [ ]:
def compute_and_assign_features(df, feat_1, feat_2):
    features_df = df.groupby(feat_1).apply(lambda group: calculate_features(group, feat_1, feat_2)).reset_index()
    return pd.merge(df, features_df, on=feat_1)

for feat_1 in ['Month']:
    for feat_2 in ['H2O Pressure (cm)']:
        # Compute features separately for train and test datasets
        train_df = compute_and_assign_features(train_df, feat_1, feat_2)
        test_df = compute_and_assign_features(test_df, feat_1, feat_2)

train_df.sort_values('Date',inplace=True)
test_df.sort_values('Date',inplace=True)

## Percentage change in Feature during the past 2 Weeks, One Month, 3 Months.


In [ ]:
periods = [10, 21, 63]
feats = ['Feature_1','Feature_2']
for feat in feats:
    for period in periods:
        train_df.loc[:, f"{feat}_PctChange_{period}"] = train_df[feat].pct_change(period)
        test_df.loc[:, f"{feat}_PctChange_{period}"] = test_df[feat].pct_change(period)

## Binning

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
n_bins = xx
est = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile') #uniform or quantile or kmeans
data = pd.concat([train_df,test_df])

feats = ['Feature_1','Feature_2']
for feat in feats:
    est.fit(data[[feat]].values)

    Xt = est.transform(train_df[[feat]].values)
    train_df[f'{feat}_Bins_Q'] = Xt.copy()

    Xt = est.transform(test_df[[feat]].values)
    test_df[f'{feat}_Bins_Q'] = Xt.copy()

## Nearest Neighbor Target

- The distance to the nearest 1
- target of 2nd nearst sample, 3rd,..
- average the top3-5 nearest
- try different distance metrics (l2,l1,cosine,manhattan’s…)
- Try Adding the nearest neighbor Features value (not only target, try to use other important featuers)

In [ ]:
from scipy.spatial.distance import cdist

def normalize(df,x):
    df[x] = df[x].fillna(df[x])
    return (df[x]-df[x].min())/df[x].max()

def NNTarget(X_train,X_valid,y_train,target):

    cols = X_train.columns

    # Concatenate Train and Test
    X_train[target] = y_train.copy()
    df = pd.concat([X_train,X_valid]).reset_index(drop=True)

    # Normalization
    d=df.copy()
    for col in cols:
        d[col]=normalize(d,col)

    # Compute distance matrix
    arr=d[cols].to_numpy()
    arr_train=d.loc[:X_train.shape[0],cols].to_numpy()
    a = cdist(arr,arr_train)

    # Exclude samples with distance 0 from our distance matrix
    a[a==0]=a.max()    

    # Get the index of the nearest samples excluding those in the above step because they cause leakage
    idx=list(np.argmin(a,axis=1))
    
    # Set the value of the feature to the target of the nearest non-null sample
    df[f'{target}_NearestNeighborTarget'] = df.reindex(index=idx)[target].values
    
    X_train[f'{target}_NearestNeighborTarget'] = df.loc[:X_train.shape[0],f'{target}_NearestNeighborTarget']
    X_valid[f'{target}_NearestNeighborTarget'] = df.loc[X_train.shape[0]:,f'{target}_NearestNeighborTarget']
    
    for n in range(2,10):
        # Get the index of the nth nearest samples excluding those in the above step because they cause leakage
        nth_idx = list(np.argsort(a, axis=1)[:, n])

        # Set the value of the feature to the target of the nth nearest non-null sample
        df[f'{target}_{n}thNearestNeighborTarget'] = df.reindex(index=nth_idx)[target].values

        X_train[f'{target}_{n}thNearestNeighborTarget'] = df.loc[:X_train.shape[0],f'{target}_{n}thNearestNeighborTarget']
        X_valid[f'{target}_{n}thNearestNeighborTarget'] = df.loc[X_train.shape[0]:,f'{target}_{n}thNearestNeighborTarget']
    
    X_train.drop(target,inplace=True,axis=1)
    
    return X_train,X_valid


In [ ]:
# # Nearest Neighbor Distance
# from scipy.spatial.distance import cdist

# def normalize(df,x):
#     return (df[x]-df[x].min())/df[x].max()

# def NNTarget(X_train,X_valid,y_train):
#     X_train, X_valid = X_train.fillna(0), X_valid.fillna(0)
#     cols = ['AB','AF','AH','AM','AR','AX','AY','AZ','BC','BD ','BN','BP','BQ','BR','CB','CC',
#             'CD ','CF','CH','CL','CR','CS','CU','CW ','DA','DE','DF','DH','DI','DL','DN','DU','DY',
#             'EB', 'EE', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI', 'FL', 'FR', 'FS', 'GB', 'GE', 'GH',
#             'GI']

#     # Concatenate Train and Test
#     X_train['Class'] = y_train.copy()
#     df = pd.concat([X_train,X_valid]).reset_index(drop=True)

#     # Normalization
#     d=df.copy()
#     for col in cols:
#         d[col]=normalize(d,col)

#     # Compute distance matrix
#     arr=d[cols].to_numpy()
#     arr_train=d.loc[:X_train.shape[0],cols].to_numpy()
    
#     # Compute distance matrix for samples with Class == 1
#     arr_train_1 = d.loc[(d['Class'] == 1) & (d.index < X_train.shape[0]), cols].to_numpy()
#     a_1 = cdist(arr, arr_train_1)
    
#     # Compute distance matrix for samples with Class == 0
#     arr_train_0 = d.loc[(d['Class'] == 0) & (d.index < X_train.shape[0]), cols].to_numpy()
#     a_0 = cdist(arr, arr_train_0)

#     # Exclude samples with distance 0 from our distance matrix
#     a_1[a_1==0]=a_1.max()    
#     a_0[a_0==0]=a_0.max()    

#     for n in range(1,11):
#         # Set the value of the feature to the distance to the nth nearest sample with Class == 1
#         df[f'{n}thNearestNeighborDistance_Class1'] = np.sort(a_1, axis=1)[:, n-1]
        
#         # Set the value of the feature to the distance to the nth nearest sample with Class == 0
#         df[f'{n}thNearestNeighborDistance_Class0'] = np.sort(a_0, axis=1)[:, n-1]
        
#         X_train[f'{n}thNearestNeighborDistance_Class1'] = df.loc[:X_train.shape[0],f'{n}thNearestNeighborDistance_Class1']
#         X_valid[f'{n}thNearestNeighborDistance_Class1'] = df.loc[X_train.shape[0]:,f'{n}thNearestNeighborDistance_Class1']
        
#         X_train[f'{n}thNearestNeighborDistance_Class0'] = df.loc[:X_train.shape[0],f'{n}thNearestNeighborDistance_Class0']
#         X_valid[f'{n}thNearestNeighborDistance_Class0'] = df.loc[X_train.shape[0]:,f'{n}thNearestNeighborDistance_Class0']
    
#     X_train.drop('Class',inplace=True,axis=1)
    
#     return X_train,X_valid


In [ ]:
# Copy this inside CV
X_Train,X_Test = NNTarget(X_Train,X_Test,y_train,'target')

## Polar Coordinates

In [ ]:
def Polar(X,y, a = 0, b = 0): # a and b represnt the center
    r = np.sqrt((X-a)**2 + (y-b)**2)
    phi = np.arctan2((y-a), (X-b))
    return r, phi

train_df['R'], train_df['Phi'] = Polar(train_df["Feature_1"],train_df["Feature_2"])
test_df['R'], test_df['Phi'] = Polar(test_df["Feature_1"],test_df["Feature_2"])


## Target Residual

In [ ]:
#If you have a feature that is very correlated with your target in a regression task, 
#fit your GBDT on the residual (target - dominating_feature) rather than target itself.
#After that add the dominating_feature to the residual to get the target
#Especially works well for forecasting problems. 
##You end up having smaller number of trees, thus decrease overfitting
#E.g. residual = target - target_mean_last_week
# Then After predicting the residual:
# Target = residual + target_mean_last_week
train_df['Target'] = train_df['Feature_the_model_depends_on_too_much_in_the_importance'] - train_df['Target']

## Bag of Words For Text Features

In [ ]:
CouVec = CountVectorizer(stop_words='english')
# CouVec = TfidfVectorizer()
df = pd.concat([train_df,test_df])
CouVec.fit(df['Feature'])
train_words = pd.DataFrame(CouVec.transform(train_df['Feature']).toarray())
test_words = pd.DataFrame(CouVec.transform(test_df['Feature']).toarray())

## Text Modeling Using LDA

In [ ]:
LDA = LatentDirichletAllocation(n_components=n_clusters, max_iter=100, random_state=42)
LDA.fit(pd.concat([train_words,test_words]))
Topics = [f'Topic_{x}' for x in range(0,n_clusters)]
train_df[Topics] = LDA.transform(train_words)
test_df[Topics] = LDA.transform(test_words)

In [ ]:
#Show The Clusters
n_top_words = 10
topic_summaries = []

# get topics and topic terms
topic_word = LDA.components_ 
vocab = CouVec.get_feature_names()

for i, topic_dist in enumerate(topic_word):
    topic_words = np.array(vocab)[np.argsort(topic_dist)][:-(n_top_words+1):-1]
    topic_summaries.append(' '.join(topic_words))
    print('Topic {}: {}'.format(i, ' | '.join(topic_words)))

## Text Features

In [ ]:
import string
import nltk
nltk.download('averaged_perceptron_tagger');

def tag_part_of_speech(text):
    text_splited = text.split(' ')
    text_splited = [''.join(c for c in s if c not in string.punctuation) for s in text_splited]
    text_splited = [s for s in text_splited if s]
    pos_list = nltk.pos_tag(text_splited)
    noun_count = len([w for w in pos_list if w[1] in ('NN','NNP','NNPS','NNS')])
    adjective_count = len([w for w in pos_list if w[1] in ('JJ','JJR','JJS')])
    verb_count = len([w for w in pos_list if w[1] in ('VB','VBD','VBG','VBN','VBP','VBZ')])
    return[noun_count, adjective_count, verb_count]

for df in ([train_df,test_df]):
  
    df['Feature_Length'] = df['Feature'].apply(lambda x : len(x))
    df['Num_Words'] = df['Feature'].apply(lambda comment: len(comment.split()))
    df['Num_Unique_Words'] = df['Feature'].apply(lambda comment: len(set(w for w in comment.split())))
    df['Words_VS_Unique'] = df['Num_Unique_Words'] / df['Num_Words']
    df['num_punctuation'] = df['Feature'].apply(lambda comment: sum(comment.count(w) for w in '\'.,;:'))
    df['nouns'], df['adjectives'], df['verbs'] = zip(*df['Feature'].apply(lambda comment: tag_part_of_speech(comment)))
    df['nouns_vs_length'] = df['nouns'] / df['Feature_Length']
    df['adjectives_vs_length'] = df['adjectives'] / df['Feature_Length']
    df['verbs_vs_length'] = df['verbs'] /df['Feature_Length']
    df['nouns_vs_words'] = df['nouns'] / df['Num_Words']
    df['adjectives_vs_words'] = df['adjectives'] / df['Num_Words']
    df['verbs_vs_words'] = df['verbs'] / df['Num_Words']
    df["count_words_title"] = df["Feature"].apply(lambda x: len([w for w in str(x).split() if w.istitle()]))
    df["mean_word_len"] = df["Feature"].apply(lambda x: np.mean([len(w) for w in str(x).split()]))
    df['punct_percent']= df['num_punctuation']*100/df['Num_Words']

In [ ]:
cols = ['flesch_reading_ease','flesch_kincaid_grade',
        'smog_index','coleman_liau_index','automated_readability_index',
        'dale_chall_readability_score','difficult_words',
        'linsear_write_formula','gunning_fog','text_standard',
        'fernandez_huerta','szigriszt_pazos','gutierrez_polini',
        'crawford']
train_df[cols] = np.nan
test_df[cols] = np.nan
for dataset in ([train_df,test_df]):  
    for i in tqdm(range(dataset.shape[0])):
        dataset['flesch_reading_ease'][i] = textstat.flesch_reading_ease(dataset['TEXT'][i])
        dataset['flesch_kincaid_grade'][i] = textstat.flesch_kincaid_grade(dataset['TEXT'][i])
        dataset['smog_index'][i] = textstat.smog_index(dataset['TEXT'][i])
        dataset['coleman_liau_index'][i] = textstat.coleman_liau_index(dataset['TEXT'][i])
        dataset['automated_readability_index'][i] = float(textstat.automated_readability_index(dataset['TEXT'][i]))
        dataset['dale_chall_readability_score'][i] = textstat.dale_chall_readability_score(dataset['TEXT'][i])
        dataset['difficult_words'][i] = textstat.difficult_words(dataset['TEXT'][i])
        dataset['linsear_write_formula'][i] = textstat.linsear_write_formula(dataset['TEXT'][i])
        dataset['gunning_fog'][i] = textstat.gunning_fog(dataset['TEXT'][i])
        dataset['text_standard'][i] = textstat.text_standard(dataset['TEXT'][i])
        dataset['fernandez_huerta'][i] = textstat.fernandez_huerta(dataset['TEXT'][i])
        dataset['szigriszt_pazos'][i] = textstat.szigriszt_pazos(dataset['TEXT'][i])
        dataset['gutierrez_polini'][i] = textstat.gutierrez_polini(dataset['TEXT'][i])
        dataset['crawford'][i] = textstat.crawford(dataset['TEXT'][i])

# Location-based Features

In [ ]:
#A 3D Encoding for the Latitude and Longitude, then pooled using mean

def lat_lon_encoding(df):
    coordinates = df[['lon', 'lat']].values

    # Encoding tricks
    emb_size = 20
    precision = 1e6

    latlon = np.expand_dims(coordinates, axis=-1)

    m = np.exp(np.log(precision)/emb_size)
    angle_freq = m ** np.arange(emb_size)
    angle_freq = angle_freq.reshape(1,1, emb_size)
    latlon = latlon * angle_freq
    latlon[..., 0::2] = np.cos(latlon[..., 0::2])

    df['lat_lon_encoding'] = np.mean(np.mean(latlon,axis=2),axis=1)
    
    return df

train_df = lat_lon_encoding(train_df)
test_df = lat_lon_encoding(test_df)

In [ ]:
#Replace Latitude and Longitude with Places names

def replace_latlon_with_places(df):
    coordinates = list(zip(df['lat'], df['lon']))
    results = rg.search(coordinates)
    df['place'] = [x['admin2'] for x in results]

    places = ['Gomoa West']

    def replace(x):
        if x in places:
            return x

        else:
            return 'Other'

    df['place'] = df['place'].apply(lambda x: replace(x))

    le = LabelEncoder()
    df['place'] = le.fit_transform(df['place'])
    
    return df

train_df = replace_latlon_with_places(train_df)
test_df = replace_latlon_with_places(test_df)

In [ ]:
# Add PCA and UMAP Represenatation for lat and lon

def PCA_UMAP_latlon(df):
    coordinates = list(zip(df['lat'], df['lon']))
    pca = PCA().fit(coordinates)
    df['pca_lat'] = pca.transform(coordinates)[:, 0]
    df['pca_lon'] = pca.transform(coordinates)[:, 1]

    umap = UMAP(n_components=2,
               n_neighbors=50,
               random_state=42).fit(coordinates)

    df['umap_lat'] = umap.transform(coordinates)[:, 0]
    df['umap_lon'] = umap.transform(coordinates)[:, 1]
    
    return df

train_df = PCA_UMAP_latlon(train_df)
test_df = PCA_UMAP_latlon(test_df)

In [ ]:
# Rotation for the Latitude and Longitude
def rotation(df):
    for angle in [15, 30, 45]:
        df[f'rot_{angle}_x'] = (np.cos(np.radians(angle)) * df['lat']) + \
                                (np.sin(np.radians(angle)) * df['lon'])
        
        df[f'rot_{angle}_y'] = (np.cos(np.radians(angle)) * df['lat']) - \
                                (np.sin(np.radians(angle)) * df['lon'])
        
    return df

train_df = rotation(train_df)
test_df = rotation(test_df)

# Missing Values

In [ ]:
#missing data percentage (Training)
total = train_df.isnull().sum().sort_values(ascending=False)
percent_1 = train_df.isnull().sum()/train_df.isnull().count()*100
percent_2 = (round(percent_1, 1)).sort_values(ascending=False)
missing_data = pd.concat([total, percent_2], axis=1, keys=['Total', '%'])
missing_data

In [ ]:
#missing data percentage (Testing)
total = test_df.isnull().sum().sort_values(ascending=False)
percent_1 = test_df.isnull().sum()/test_df.isnull().count()*100
percent_2 = (round(percent_1, 1)).sort_values(ascending=False)
missing_data = pd.concat([total, percent_2], axis=1, keys=['Total', '%'])
missing_data

In [ ]:
# Choice 1: impute with mode/median/mean of each/both dataset

# df = pd.concat([train_df,test_df])
for dataset in (train_df,test_df):
    dataset = dataset.fillna(dataset.mode(axis=0).loc[0])  #mode
#     dataset = dataset.fillna(dataset.median())  #median/mean
    
#-----------------------------------------------------------------------------------------------------------------------------    
# Choice 2: imputing based on other feature of each/both dataset

# df = pd.concat([train_df,test_df])
feats = train_df.columns[train_df.isna().any()].tolist()
for feat in feats:
    train_df[feat] = train_df[feat].fillna(train_df.groupby('Feature')[feat].transform('median')) #median/mean
#     train_df[feat] = train_df[feat].fillna(train_df.groupby('Feature')[feat].transform(lambda x: x.value_counts().idxmax())) #mode

feats = test_df.columns[test_df.isna().any()].tolist()
for feat in feats:
    test_df[feat] = test_df[feat].fillna(test_df.groupby('Feature')[feat].transform('median')) #median/mean
#     test_df[feat] = test_df[feat].fillna(test_df.groupby('Feature')[feat].transform(lambda x: x.value_counts().idxmax())) #mode

#-----------------------------------------------------------------------------------------------------------------------------
# Choice 3: Bfill/Ffill

train_df = train_df.ffill().bfill()   
test_df = test_df.ffill().bfill()

# Encoding

##### Label Encoding (Order)

In [ ]:
feats = list(train_df.select_dtypes(include=['object','category']).columns)
# feats.remove('Target')
le = LabelEncoder()
df = pd.concat([train_df, test_df])
for f in feats:
    le.fit(df[f])
    train_df[f] = le.transform(train_df[f])
    test_df[f] = le.transform(test_df[f])

##### Frequency Encoding

In [ ]:
feats = list(test_df.select_dtypes(include=['object','category']).columns)
df = pd.concat([train_df, test_df])
for feat in feats:
    count_encoder = ce.CountEncoder(cols=feat)
    count_encoder.fit(df[feat])
    train_df[f'{feat}_count'] = count_encoder.transform(train_df[feat])
    test_df[f'{feat}_count'] = count_encoder.transform(test_df[feat])

##### One-Hot Encoding

In [ ]:
feats = list(test_df.select_dtypes(include=['object','category']).columns)
for feat in feats: 
    Names = [f'{feat}_{x}' for x in train_df[feat].value_counts().keys().sort_values()]
    OHE_cols = pd.DataFrame(pd.get_dummies(train_df[feat]).values,index = train_df.index, columns = Names)
    train_df = pd.concat([train_df,OHE_cols],axis=1)
    
    Names = [f'{feat}_{x}' for x in test_df[feat].value_counts().keys().sort_values()]
    OHE_cols = pd.DataFrame(pd.get_dummies(test_df[feat]).values,index = test_df.index, columns = Names)
    test_df = pd.concat([test_df,OHE_cols],axis=1)
    
    # Add columns for unique values in train but not in test
    unique_train_cols = set(train_df.columns) - set(test_df.columns)
    for col in unique_train_cols:
        test_df.loc[:,col] = 0

##### Target Encoding

In [ ]:
from sklearn import base
from sklearn.model_selection import KFold
class KFoldTargetEncoderTrain(base.BaseEstimator, base.TransformerMixin):

    def __init__(self,colnames,targetName,n_fold=5,verbosity=True,discardOriginal_col=False):

        self.colnames = colnames
        self.targetName = targetName
        self.n_fold = n_fold
        self.verbosity = verbosity
        self.discardOriginal_col = discardOriginal_col


    def fit(self, X, y=None):
        return self


    def transform(self,X,test_df):

        assert(type(self.targetName) == str)
        assert(type(self.colnames) == str)
        assert(self.colnames in X.columns)
        assert(self.targetName in X.columns)

        mean_of_target = X[self.targetName].mean()
        kf = KFold(n_splits = self.n_fold)



        col_mean_name = self.colnames + '_' + 'Kfold_Target_Enc'
        X[col_mean_name] = np.nan

        for tr_ind, val_ind in kf.split(X):
            X_tr, X_val = X.iloc[tr_ind], X.iloc[val_ind]
            print(tr_ind,val_ind)
            X.loc[X.index[val_ind],col_mean_name] = X_val[self.colnames].map(X_tr.groupby(self.colnames)[self.targetName].mean()).values

        X[col_mean_name].fillna(mean_of_target, inplace = True)
        
        test_df[col_mean_name] = test_df[self.colnames].map(X.groupby(self.colnames)[col_mean_name].mean())
        
        if self.verbosity:

            encoded_feature = X[col_mean_name].values
            print('Correlation between the new feature, {} and, {} is {}.'.format(col_mean_name,
                                                                                      self.targetName,
                                                                                      np.corrcoef(X[self.targetName].values, encoded_feature)[0][1]))
        if self.discardOriginal_col:
            X = X.drop(self.targetName, axis=1)
            

        return X, test_df

In [ ]:
#These Features should not have NaNs and Should be label encoded
feats = ['Feature_1','Feature_2']
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for feat in feats:
    tar_enc = KFoldTargetEncoderTrain(feat,'Target',n_fold=5)
    tar_enc.fit(train_df)
    train_df, test_df = tar_enc.transform(train_df,test_df)

## Dropping Duplicates and Constants Features

In [ ]:
print('Features Before Dropping: ', train_df.shape)
cols = train_df.columns
dup = []
for feat_1 in tqdm(cols):
    if (feat_1 in dup):
        continue
    for feat_2 in cols.drop(feat_1):
        if (feat_2 in dup):
            continue
        if (train_df[feat_1].equals(train_df[feat_2])):
            train_df.drop(feat_2,inplace=True,axis=1)
            test_df.drop(feat_2,inplace=True,axis=1)
            dup.append(feat_2)

In [ ]:
for feat in tqdm(test_df.columns):
    if ((len(train_df[feat].value_counts().keys()) == 1) | (len(test_df[feat].value_counts().keys()) == 1)):
        train_df.drop(feat,inplace=True,axis=1)
        test_df.drop(feat,inplace=True,axis=1)
print('Features After Dropping: ', train_df.shape)

# Modeling

In [ ]:
# Binning the target
# train_df['Target_binned'] = pd.qcut(train_df['Target'], q=10, labels=False)

In [ ]:
lg_params = {'max_depth': 6}
xg_params = {'max_depth': 6}
cb_params = {'depth': 6, 'iterations': 1000, 'learning_rate': 0.01}
rf_params = {'max_depth': 15, 'n_jobs': -1, 'n_estimators': 1000}
elastic_params = {'alpha':200.0, 'l1_ratio':0.25, 'max_iter':10000}

In [ ]:
# lgbm = LGBMClassifier(**lg_params, random_state=42)
lgbm = LGBMRegressor(**lg_params, random_state=42)
llgbm = TransformedTargetRegressor(lgbm, func = np.log1p, inverse_func = np.expm1)

# xg = XGBClassifier(**xg_params, random_state=42)
xg = XGBRegressor(**xg_params, random_state=42)
lxg = TransformedTargetRegressor(xg, func = np.log1p, inverse_func = np.expm1)

# cb = CatBoostClassifier(**cb_params, random_state=42)
cb = CatBoostRegressor(**cb_params, random_state=42)
lcb = TransformedTargetRegressor(cb, func = np.log1p, inverse_func = np.expm1)

# rf = RandomForestClassifier(**rf_params, random_state=42)
rf = RandomForestRegressor(**rf_params, random_state=42)
lrf = TransformedTargetRegressor(rf, func = np.log1p, inverse_func = np.expm1)

elastic = ElasticNet(**elastic_params, random_state=42)

In [ ]:
models = [
            ('lg', lgbm),
            ('xg', xg),
            ('cb', cb),
            ('rf', rf),
             ]

model =  VotingRegressor(estimators=models,verbose = True, n_jobs = -1)
# model =  VotingClassifier(estimators=models, voting='soft', flatten_transform=True,verbose = True, n_jobs = -1)

model =  BaggingRegressor(my_model, n_estimators=500, max_samples=100, bootstrap=True, n_jobs=-1, random_state = 42,verbose = True)
# model =  BaggingClassifier(my_model, n_estimators=500, max_samples=100, bootstrap=True, n_jobs=-1, random_state = 42,verbose = True)

model =  StackingRegressor(estimators=models, final_estimator=LinearRegression(), cv=5,verbose = True, n_jobs = -1)
# model =  StackingClassifier(estimators=models, final_estimator=LogisticRegression(), cv=5,verbose = True, n_jobs = -1)

## Adversarial Validation

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
from xgboost import plot_importance
from xgboost import cv

X_train = train_df.drop(['Target'], axis=1)
X_test = test_df

# add the train/test labels
X_train["AV_label"] = 0
X_test["AV_label"]  = 1

# make one big dataset
df = pd.concat([X_train, X_test], axis=0, ignore_index=True)

# shuffle
df = df.sample(frac=1)

# create our DMatrix (the XGBoost data structure)
X = df.drop(['AV_label'], axis=1)
y = df['AV_label']
XGBdata = xgb.DMatrix(data=X,label=y)

# our XGBoost parameters
params = {"objective":"binary:logistic",
          "eval_metric":"logloss",
          'learning_rate': 0.05,
          'max_depth': 5,
          'tree_method':'gpu_hist', 
          'predictor':'gpu_predictor'}

# perform cross validation with XGBoost
cross_val_results = cv(dtrain=XGBdata, params=params, 
                       nfold=5, metrics="auc", 
                       num_boost_round=200,early_stopping_rounds=20,
                       as_pandas=True)

# print out the final result
print((cross_val_results["test-auc-mean"]).tail(1))

In [ ]:
classifier = XGBClassifier(eval_metric='logloss',use_label_encoder=False,tree_method='gpu_hist', predictor='gpu_predictor')
classifier.fit(X, y)
plotImp(classifier,df.drop('Target',axis=1))

## Validation:

In [ ]:
scores = []

##### First Option

In [ ]:
train = train_df[train_df['date'] <= '2016-03-27']
vali = train_df[(train_df['date'] > '2016-03-27') & (train_df['date'] <= '2016-04-24')]
y_Test = vali['Target'].copy()
vali.drop('Target',inplace=True,axis=1)

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

In [ ]:
print('Validating...')

# cb.fit(train.drop('Target',axis=1),train['Target'],eval_set=[(vali, y_test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
llgbm.fit(train.drop('Target',axis=1),train['Target'])
y_pred = llgbm.predict(vali)
#Error Analysis
# y_pred = lgbm.predict_proba(X_Test)
err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
# err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification

score = mean_absolute_error(y_Test,y_pred)
print(score)
scores.append(score)

if (scores[-1] > scores[-2]):
    print('Worse')
elif (scores[-1] < scores[-2]):
    print('Better')
else:
    print('Same')

##### Second Option

In [ ]:
cutoff = train_df.date.max() - pd.to_timedelta(28, unit = 'D')
train = train_df.loc[train_df.date < cutoff].copy()
vali = train_df.loc[train_df.date >= cutoff].copy()
y_Test = vali['Target'].copy()
vali.drop('Target',inplace=True,axis=1)

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

In [ ]:
print('Validating...')

# cb.fit(train.drop('Target',axis=1),train['Target'],eval_set=[(vali, y_test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
llgbm.fit(train.drop('Target',axis=1),train['Target'])
y_pred = llgbm.predict(vali)
#Error Analysis
# y_pred = lgbm.predict_proba(X_Test)
err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
# err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification

score = mean_absolute_error(y_Test,y_pred)
print(score)
scores.append(score)

if (scores[-1] > scores[-2]):
    print('Worse')
elif (scores[-1] < scores[-2]):
    print('Better')
else:
    print('Same')

##### Third Option

In [ ]:
print('Validating...')

Means = []
STDs = []

X = train_df.drop('Target',axis=1).values
y = train_df['Target'].values

temp = train_df.copy()
temp['date'] = temp.index
temp = temp.reset_index(drop=True)

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

scores = []
for train_index, test_index in GroupTimeSeriesSplit().split(X, y, groups=temp['date'].values):
    X_Train, X_Test = X[train_index], X[test_index]
    y_Train, y_Test = y[train_index], y[test_index]
    # cb.fit(X_Train,y_Train,eval_set=[(X_Test, y_Test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
    lgbm.fit(X_Train,y_Train)
    y_pred = lgbm.predict(X_Test)
    #Error Analysis
#     y_pred = lgbm.predict_proba(X_Test)
    err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
#     err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification
    scores.append(mean_absolute_error(y_Test,y_pred))
    print(scores[-1])
    
print("\nMean:",np.mean(scores),"\nSTD: ", np.std(scores))
Means.append(np.mean(scores))
STDs.append(np.std(scores))

if (Means[-1] > Means[-2]):
    print('Better')
elif (Means[-1] < Means[-2]):
    print('Worse')
else:
    print('Same')

if (STDs[-1] > STDs[-2]):
    print('Worse')
elif (STDs[-1] < STDs[-2]):
    print('Better')
else:
    print('Same')

##### Fourth Option

In [ ]:
print('Validating...')

Means = []
STDs = []

X = train_df.drop('Target',axis=1).values
y = train_df['Target'].values

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

scores = []                   #Or StratifiedShuffleSplit
for train_index, test_index in StratifiedKFold(n_splits=5).split(X, y):
    X_Train, X_Test = X[train_index], X[test_index]
    y_Train, y_Test = y[train_index], y[test_index]
    # cb.fit(X_Train,y_Train,eval_set=[(X_Test, y_Test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
    lgbm.fit(X_Train,y_Train)
    y_pred = lgbm.predict(X_Test)
    #Error Analysis
#     y_pred = lgbm.predict_proba(X_Test)
    err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
#     err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification
    scores.append(mean_absolute_error(y_Test,y_pred))
    print(scores[-1])
    
print("\nMean:",np.mean(scores),"\nSTD: ", np.std(scores))
Means.append(np.mean(scores))
STDs.append(np.std(scores))

if (Means[-1] > Means[-2]):
    print('Better')
elif (Means[-1] < Means[-2]):
    print('Worse')
else:
    print('Same')

if (STDs[-1] > STDs[-2]):
    print('Worse')
elif (STDs[-1] < STDs[-2]):
    print('Better')
else:
    print('Same')

##### Fifth Option

In [ ]:
print('Validating...')

Means = []
STDs = []

X = train_df.drop('Target',axis=1).values
y = train_df['Target'].values

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

scores = []
for train_index, test_index in KFold(n_splits=5).split(X, y):
    X_Train, X_Test = X[train_index], X[test_index]
    y_Train, y_Test = y[train_index], y[test_index]
    # cb.fit(X_Train,y_Train,eval_set=[(X_Test, y_Test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
    lgbm.fit(X_Train,y_Train)
    y_pred = lgbm.predict(X_Test)
    #Error Analysis
#     y_pred = lgbm.predict_proba(X_Test)
    err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
#     err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification
    scores.append(mean_absolute_error(y_Test,y_pred))
    print(scores[-1])
    
print("\nMean:",np.mean(scores),"\nSTD: ", np.std(scores))
Means.append(np.mean(scores))
STDs.append(np.std(scores))

if (Means[-1] > Means[-2]):
    print('Better')
elif (Means[-1] < Means[-2]):
    print('Worse')
else:
    print('Same')

if (STDs[-1] > STDs[-2]):
    print('Worse')
elif (STDs[-1] < STDs[-2]):
    print('Better')
else:
    print('Same')

### NOTE: The difference between GroupKFold and StratifiedGroupKFold is that the former attempts to create balanced folds such that the number of distinct groups is approximately the same in each fold, whereas StratifiedGroupKFold attempts to create folds which preserve the percentage of samples for each class as much as possible given the constraint of non-overlapping groups between splits.

###### Sixth Option

In [ ]:
print('Validating...')

Means = []
STDs = []

X = train_df.drop('Target',axis=1).values
y = train_df['Target'].values

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

scores = []
for train_index, test_index in StratifiedGroupKFold(n_splits=5).split(X, y):
    X_Train, X_Test = X[train_index], X[test_index]
    y_Train, y_Test = y[train_index], y[test_index]
    # cb.fit(X_Train,y_Train,eval_set=[(X_Test, y_Test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
    lgbm.fit(X_Train,y_Train)
    y_pred = lgbm.predict(X_Test)
    #Error Analysis
#     y_pred = lgbm.predict_proba(X_Test)
    err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
#     err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification
    scores.append(mean_absolute_error(y_Test,y_pred))
    print(scores[-1])
    
print("\nMean:",np.mean(scores),"\nSTD: ", np.std(scores))
Means.append(np.mean(scores))
STDs.append(np.std(scores))

if (Means[-1] > Means[-2]):
    print('Better')
elif (Means[-1] < Means[-2]):
    print('Worse')
else:
    print('Same')

if (STDs[-1] > STDs[-2]):
    print('Worse')
elif (STDs[-1] < STDs[-2]):
    print('Better')
else:
    print('Same')

###### Seventh Option

In [ ]:
print('Validating...')

Means = []
STDs = []

X = train_df.drop('Target',axis=1).values
y = train_df['Target'].values

err_anal = train_df.reset_index(drop=True).copy()
err_anal['y_pred'] = np.nan  #Regression
# err_anal[Target_Values] = np.nan #Classification

scores = []
for train_index, test_index in GroupKFold(n_splits=5).split(X, y):
    X_Train, X_Test = X[train_index], X[test_index]
    y_Train, y_Test = y[train_index], y[test_index]
    # cb.fit(X_Train,y_Train,eval_set=[(X_Test, y_Test)], cat_features=cat_feats, verbose=100)  #Early Stopping for Catboost
    lgbm.fit(X_Train,y_Train)
    y_pred = lgbm.predict(X_Test)
    #Error Analysis
#     y_pred = lgbm.predict_proba(X_Test)
    err_anal.loc[err_anal.index[test_index],'y_pred'] = y_pred    #Regression
#     err_anal.loc[err_anal.index[test_index],Target_Values] = y_pred   #Classification
    scores.append(mean_absolute_error(y_Test,y_pred))
    print(scores[-1])
    
print("\nMean:",np.mean(scores),"\nSTD: ", np.std(scores))
Means.append(np.mean(scores))
STDs.append(np.std(scores))

if (Means[-1] > Means[-2]):
    print('Better')
elif (Means[-1] < Means[-2]):
    print('Worse')
else:
    print('Same')

if (STDs[-1] > STDs[-2]):
    print('Worse')
elif (STDs[-1] < STDs[-2]):
    print('Better')
else:
    print('Same')

## Show the Features Importances 

In [ ]:
plotImp(lgbm,train_df.drop('Sales',axis=1))

## Permutation Importance

In [ ]:
import pickle
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from colorama import Fore, Style

print('Validating...')
X = train_df.drop(['utility_agent1', 'num_wins_agent1', 'num_draws_agent1', 'num_losses_agent1'], axis=1)
y = train_df['utility_agent1'].values  

oof_preds = np.zeros((len(train_df),))
scores = []
importances_list = []

# Cross-validation loop
kf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_index, test_index) in tqdm(enumerate(kf.split(X, y.astype(str), groups=groups))):
    X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
    y_Train, y_Test = y[train_index], y[test_index]

    # Fit model
    cb.fit(X_Train, y_Train - X_Train['AdvantageP1'])

    # Predict and store out-of-fold predictions
    y_pred = np.clip((cb.predict(X_Test) + X_Test['AdvantageP1']), -1, 1)
    oof_preds[test_index] = y_pred

    # Calculate permutation importance
    result = permutation_importance(cb, X_Test, y_Test - X_Test['AdvantageP1'], scoring='neg_root_mean_squared_error', 
                                    n_repeats=2, n_jobs=64, random_state=42)

    print(f"{Fore.GREEN}{Style.BRIGHT}Important features: {(result['importances_mean'] > 0).mean():.0%}   {Style.RESET_ALL}")
    
    # Store the importances for this fold
    importance_df = pd.DataFrame({
        'importance': result['importances_mean'],
        'std': result['importances_std']
    }, index=X_Test.columns).sort_values('importance', ascending=False)
    

    # Append this fold's importance to the list
    importances_list.append(importance_df['importance'])

# Compute the average importance across all folds
mean_importance = pd.concat(importances_list, axis=1).mean(axis=1).sort_values(ascending=False)

# Display the averaged importances
mean_importance_df = pd.DataFrame({
    'Name': mean_importance.index,
    'importance': mean_importance
}).sort_values('importance', ascending=False)
mean_importance_df.to_csv('imps.csv',index=False)
display(mean_importance_df.head(50))


## Postprocessing trick for SMAPE metric

In [ ]:
# Step 1: Multiclass Classification
params = {
        'boosting_type': 'gbdt',
        'objective': 'multiclass',
...
}

In [ ]:
# Step 2: Train model (could be xgboost catboost lgbm)
estimator.train(...)

In [ ]:
# Step 3: Predict probabilities:
probas = estimator.predict(...)

In [ ]:
# Step 4: Make direct SMAPE1P optimization from probabilities
def single_smape1p(preds, tgt):
    x = np.tile(np.arange(preds.shape[1]), (preds.shape[0], 1))
    x = np.abs(x - tgt) / (2 + x + tgt)
    return (x * preds).sum(axis=1)

def opt_smape1p(preds):
    x = np.hstack([single_smape1p(preds, i).reshape(-1,1) for i in range(preds.shape[1])])
    return x.argmin(axis=1)

final_predictions = opt_smape1p(probas)

## Postporcessing Nelder-Mead

In [ ]:
from scipy.optimize import minimize

def f(x, thresholds, n):
    pred1 = y_pred.copy()
    for i in range(n):
        if i == 0:
            pred1[y_pred >= thresholds[i]] = y_pred[y_pred >= thresholds[i]] * x[i]
        elif i == n-1:
            pred1[y_pred < thresholds[i-1]] = y_pred[y_pred < thresholds[i-1]] * x[i]
        else:
            pred1[(y_pred < thresholds[i-1]) & (y_pred >= thresholds[i])] = y_pred[(y_pred < thresholds[i-1]) & (y_pred >= thresholds[i])] * x[i]
    # Choose the competition metric
    score = np.sqrt(mean_squared_error(y, pred1))
    return score

# Define the thresholds and the number of bins
# 1- For Probabilities
thresholds = np.linspace(0.95,0.05,20)
# 2- For Regression
thresholds = [100,50,30,10,-20,-50,....]
n = len(thresholds) + 1

# Perform the optimization
result = minimize(lambda x: f(x, thresholds, n), [1]*n, method="Nelder-Mead")

In [ ]:
# Get the optimal coefficients from the result object
optimal_coefficients = result.x

# Apply the optimal coefficients to the test predictions
pred1 = y_test.copy()
for i in range(n):
    if i == 0:
        pred1[y_test >= thresholds[i]] = y_test[y_test >= thresholds[i]] * optimal_coefficients[i]
    elif i == n-1:
        pred1[y_test < thresholds[i-1]] = y_test[y_test < thresholds[i-1]] * optimal_coefficients[i]
    else:
        pred1[(y_test < thresholds[i-1]) & (y_test >= thresholds[i])] = y_test[(y_test < thresholds[i-1]) & (y_test >= thresholds[i])] * optimal_coefficients[i]

pred1

## Ensembling: Hill Climbing

In [ ]:
# metric = mean_squared_error

# def ensemble(models, weights, X):
#     predictions = np.zeros_like(models[0].predict(X))
#     for model, weight in zip(models, weights):
#         predictions += weight * model.predict(X)
#     return predictions

# def hill_climbing(models, X, y, step_size=0.1, max_iterations=1000):
#     weights = np.ones(len(models)) / len(models)
#     current_predictions = ensemble(models, weights, X)
#     current_error = metric(y, current_predictions)
#     for i in range(max_iterations):
#         best_weights = weights.copy()
#         best_error = current_error
#         for j in range(len(weights)):
#             for direction in [-1, 1]:
#                 new_weights = weights.copy()
#                 new_weights[j] += direction * step_size
#                 new_weights = np.maximum(new_weights, 0)
#                 new_weights /= np.sum(new_weights)
#                 new_predictions = ensemble(models, new_weights, X)
#                 new_error = metric(y, new_predictions)
#                 if new_error < best_error:
#                     best_weights = new_weights
#                     best_error = new_error
#         if best_error == current_error:
#             break
#         weights = best_weights
#         current_error = best_error
#     return weights


In [ ]:
# # Compute the optimal weights for the ensemble using hill climbing
# weights = hill_climbing(models, X_Train, y_Train)
# # Make predictions with the ensemble on the test data
# predictions = ensemble(models, weights, X_Test)
# # Evaluate the performance of the ensemble
# error = metric(y_Test, predictions)

## Inference

* Try Diversifying the input and use Bagging
* XGBoost outperform LGBM
* Train another model on the same period of the test data but from the past years (e.g. if the test data in October 2021, train another model on October 2020,2019...) then ensemble it with the original one
* Consider use different types of mean (Harmonic mean...) in the last layer of stacking 
* Multiply by a correction factor
* Average the seeds
* If the problem is regression try the following ensembing idea:
* 1- normalize the target to be between 0 and 1
* 2- Fit a binary classification model the predict_proba
* 3- Ensemble it with your regression model to get a boost
* 4- OOF Predictions
* 5- Instead of OOF Predictions (80% of the data each fold) save the hyperparamters and train the final model on the whole dataset with the parameters learned from during the validation.

##### Regression/Classification (Categories)

In [ ]:
X = train_df.drop('Target',axis=1)
y = train_df['Target']

model.fit(X,y)
Predictions = model.predict(test_df)

Predictions = pd.DataFrame(Predictions, index = test_df.index)
# Predictions = Predictions.transform(lambda x: mapping_rev[x])   #If Original Classes are Needed
submission = pd.DataFrame({"ID": ID ,"Target": Predictions.values})
submission.to_csv('Submission.csv',index=False)

##### Classification (Probabilities)

In [ ]:
X = train_df.drop('Target',axis=1)
y = train_df['Target']

model.fit(X,y)
Predictions = model.predict_proba(test_df)

submission = pd.DataFrame({"Id": ID})
Predictions = pd.DataFrame(Predictions, index = test_df.index, columns = Target_Values)
submission = pd.concat([submission.reset_index(drop=True),Predictions.reset_index(drop=True)],axis=1)
submission.to_csv('Submission.csv',index=False)

##### Avg Seeds

In [ ]:
# Predictions = pd.DataFrame()
# X = train_df.drop('Target',axis=1)
# y = train_df['Target']

# for seed in range(30,46):
#     lgbm = LGBMRegressor(**params, random_state=seed)
#     llgbm = TransformedTargetRegressor(lgbm, func = np.log1p, inverse_func = np.expm1)
#     llgbm.fit(X, y)

#     Predictions[f'Target_{seed}'] = llgbm.predict(test_df)
#     Predictions[f'Target_{seed}'] = Predictions[f'Target_{seed}'] * 0.995
# Predictions['Mean'] = Predictions.mean(axis=1)
# Predictions['HMean'] = Predictions.apply(stats.hmean, axis=1)
# Predictions['GMean'] = Predictions.apply(stats.gmean, axis=1)

In [ ]:
# FinalPred = Predictions[['Mean','HMean','GMean','Mean_2','HMean_2','GMean_2']].apply(stats.hmean,axis=1)

In [ ]:
# submission = pd.DataFrame({"Id": ID ,"Sales": FinalPred.values})
# submission.to_csv('FinalSubmission.csv',index=False)